<div align="center">

# Notebook 3

# Augmentation de données — Classification multi-classes FOCH

**Notebook `3_0` — Augmentation contrôlée et préservant le label (WavLM-Large)**

![Python](https://img.shields.io/badge/Python-3.14-blue?style=flat-square&logo=python)
![PyTorch](https://img.shields.io/badge/PyTorch-2.x-EE4C2C?style=flat-square&logo=pytorch)
![WavLM](https://img.shields.io/badge/🤗-WavLM--Large-FFD21F?style=flat-square)

</div>

Ce notebook prolonge le baseline `2_0_foch_test.ipynb` (qui reste le **groupe de
contrôle**) en y ajoutant trois niveaux d'augmentation, évalués par **ablation** :

| Run | Contenu | `RUN_NAME` |
|-----|---------|-----------|
| **E0** | Baseline (déjà calculé dans `2_0`) | `wavlm_large_baseline` |
| **E1** | + augmentation signal-level | `aug_signal` |
| **E2** | + signal-level + SpecAugment | `aug_signal_specaug` |
| **E3** | + signal-level + SpecAugment + SeedVC | `aug_full` |

>  **Principe directeur** — en pathologie vocale, les marqueurs de classe
> (jitter, shimmer, HNR, irrégularité de F0) résident dans la **source glottique**.
> Toute augmentation doit donc être **vérifiée** (harnais parselmouth) pour
> garantir qu'elle ne détruit pas le label. L'ensemble de **validation reste
> strictement composé d'enregistrements originaux**, identiques au baseline.

## 1. Configuration et imports

On réutilise la logique éprouvée de `2_0` via le module partagé **`foch_utils.py`**
(parsing, découpage groupé par patient, Dataset, modèle, entraînement, évaluation),
ce qui évite toute duplication ou dérive de comportement. On ajoute ici les chemins
et hyperparamètres propres à l'augmentation.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 1 — Configuration et imports
# ═══════════════════════════════════════════════════════════════════════════
import os, glob, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torchaudio
import soundfile as sf

import foch_utils as F   # module partagé (logique 2_0)

warnings.filterwarnings('ignore')

# ── Reproductibilité ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
RNG = np.random.default_rng(SEED)

# ── FFmpeg (backend de décodage audio) ───────────────────────────────────────
import shutil
_ffmpeg_bin = r'C:/Users/AdminIA/Downloads/ffmpeg-8.1.1-full_build-shared/ffmpeg-8.1.1-full_build-shared/bin'
if os.path.isdir(_ffmpeg_bin):
    os.environ['PATH'] += os.pathsep + _ffmpeg_bin

# ── Chemins (identiques à 2_0) ───────────────────────────────────────────────
CSV_PATH        = r'C:/Users/AdminIA/Desktop/ORL_IA_FOCH_Callum_HOLLIDAY/csv_best/orl_df_vowel_master.csv'
FOCH_AUDIO_DIR  = r'C:/Users/AdminIA/Desktop/ORL_IA_FOCH_Callum_HOLLIDAY/Segmented_Voyelles_Best'
SVD_HEALTHY_DIR = r'C:/Users/AdminIA/Documents/SVD/healthy'
MODEL_DIR       = r'C:/Users/AdminIA/Documents/models/wavlm-large'
WEIGHTS_DIR     = r'C:/Users/AdminIA/Documents/models/.poids_modèles'
METRICS_DIR     = r'C:/Users/AdminIA/Documents/models/.foch_métriques'

# ── Nouveaux artefacts d'augmentation ────────────────────────────────────────
AUG_AUDIO_DIR   = r'C:/Users/AdminIA/Desktop/ORL_IA_FOCH_Callum_HOLLIDAY/Segmented_Voyelles_Augmented'
AUG_SIGNAL_DIR  = os.path.join(AUG_AUDIO_DIR, 'signal')
AUG_VC_DIR      = os.path.join(AUG_AUDIO_DIR, 'vc')
for d in (WEIGHTS_DIR, METRICS_DIR, AUG_SIGNAL_DIR, AUG_VC_DIR):
    os.makedirs(d, exist_ok=True)

# ── Classes & hyperparamètres ────────────────────────────────────────────────
TARGET_PATHOLOGIES = ['PR', 'LMB', 'fuite glottique']
HEALTHY_LABEL      = 'Healthy'
CLASSES            = TARGET_PATHOLOGIES + [HEALTHY_LABEL]
NUM_LABELS         = len(CLASSES)
N_HEALTHY          = 100

TARGET_SR    = 16000
MAX_LEN_S    = 4.0
BATCH_SIZE   = 4
ACCUM_STEPS  = 4
EPOCHS       = 30            # réduire pour un test rapide du pipeline
LR           = 1e-5
PATIENCE     = 6
FREEZE_FE    = True

# Augmentation
TARGET_PER_CLASS = 80        # effectif d'entraînement visé par classe (équilibrage)
SPECAUG_TIME_PROB    = 0.08  # masquage temporel interne WavLM (SpecAugment)
SPECAUG_FEATURE_PROB = 0.05

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Configuration chargée.')
print(f'  Classes : {CLASSES}')
print(f'  Device  : {device}')
print(f'  Cible/classe (train) : {TARGET_PER_CLASS}')

Configuration chargée.
  Classes : ['PR', 'LMB', 'fuite glottique', 'Healthy']
  Device  : cuda
  Cible/classe (train) : 80


## 2. Reconstruction du jeu et découpage identique au baseline

On reconstruit le DataFrame et le découpage train/validation **exactement** comme
dans `2_0` (mêmes opérations, même `SEED`). C'est crucial : le `val_df` doit rester
**identique** pour que la comparaison E0 → E3 soit honnête. On annote chaque ligne
avec des colonnes de **provenance** (`is_aug`, `aug_type`, `source_filepath`) qui
serviront à tracer et à prévenir toute fuite.

In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 2 — Reconstruction du jeu + split identique au baseline
# ═══════════════════════════════════════════════════════════════════════════
df = F.build_base_dataframe(CSV_PATH, FOCH_AUDIO_DIR, SVD_HEALTHY_DIR,
                            TARGET_PATHOLOGIES, HEALTHY_LABEL, CLASSES,
                            N_HEALTHY, SEED)
df, label2id, id2label = F.add_labels(df, CLASSES)
train_df, val_df = F.split_train_val(df, SEED)

# Colonnes de provenance (les originaux ne sont pas augmentés)
for d in (train_df, val_df):
    d['is_aug'] = False
    d['aug_type'] = 'original'
    d['source_filepath'] = d['filepath']

print(f'Train original : {len(train_df)} | Val (fixe, originaux) : {len(val_df)}')
print('\nTrain par classe :')
print(train_df['label_str'].value_counts().reindex(CLASSES).to_string())
print('\nVal par classe :')
print(val_df['label_str'].value_counts().reindex(CLASSES).to_string())

# Processor WavLM (extracteur de features Wav2Vec2)
from transformers import Wav2Vec2FeatureExtractor
processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_DIR)
print('\n✓ Processor chargé')

Train original : 197 | Val (fixe, originaux) : 49

Train par classe :
label_str
PR                 58
LMB                38
fuite glottique    21
Healthy            80

Val par classe :
label_str
PR                 14
LMB                10
fuite glottique     5
Healthy            20

✓ Processor chargé


## 3. Harnais de validation acoustique (parselmouth)

Avant d'intégrer toute augmentation, on se dote d'un **instrument de mesure** des
marqueurs de pathologie issus de l'analyse Praat (via `parselmouth`) :

- **Jitter (local)** — instabilité cycle-à-cycle de la fréquence fondamentale ;
- **Shimmer (local)** — instabilité cycle-à-cycle de l'amplitude ;
- **HNR** — rapport harmoniques/bruit (souffle, composante apériodique) ;
- **F0** moyen et écart-type.

Ces grandeurs *définissent* en grande partie les classes pathologiques. Si une
augmentation les déplace fortement, elle altère le label : c'est précisément ce
que ce harnais permet de **quantifier** (avant/après).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 3 — Harnais parselmouth : mesure des marqueurs de source glottique
# ═══════════════════════════════════════════════════════════════════════════
import parselmouth
from parselmouth.praat import call

def _load_mono_16k(path, sr=TARGET_SR):
    wav, osr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(0, keepdim=True)
    if osr != sr:
        wav = torchaudio.transforms.Resample(osr, sr)(wav)
    return wav.squeeze().numpy().astype('float64')

def mesurer_marqueurs(wav_or_path, sr=TARGET_SR):
    "Renvoie {jitter, shimmer, hnr, f0_mean, f0_std} ; NaN si non mesurable."
    x = _load_mono_16k(wav_or_path, sr) if isinstance(wav_or_path, str) else np.asarray(wav_or_path, dtype='float64')
    out = {'jitter': np.nan, 'shimmer': np.nan, 'hnr': np.nan, 'f0_mean': np.nan, 'f0_std': np.nan}
    try:
        snd = parselmouth.Sound(x, sampling_frequency=sr)
        pp = call(snd, 'To PointProcess (periodic, cc)', 75, 500)
        out['jitter']  = call(pp, 'Get jitter (local)', 0, 0, 1e-4, 0.02, 1.3)
        out['shimmer'] = call([snd, pp], 'Get shimmer (local)', 0, 0, 1e-4, 0.02, 1.3, 1.6)
        harm = call(snd, 'To Harmonicity (cc)', 0.01, 75, 0.1, 1.0)
        out['hnr'] = call(harm, 'Get mean', 0, 0)
        pitch = call(snd, 'To Pitch', 0.0, 75, 500)
        out['f0_mean'] = call(pitch, 'Get mean', 0, 0, 'Hertz')
        out['f0_std']  = call(pitch, 'Get standard deviation', 0, 0, 'Hertz')
    except Exception as e:
        pass
    return out

def comparer_marqueurs(paths_avant, wavs_apres, sr=TARGET_SR):
    "Compare les distributions avant (chemins) / apres (formes d'onde) ; renvoie un DataFrame moyen + deltas."
    av = pd.DataFrame([mesurer_marqueurs(p, sr) for p in paths_avant]).mean()
    ap = pd.DataFrame([mesurer_marqueurs(w, sr) for w in wavs_apres]).mean()
    comp = pd.DataFrame({'avant': av, 'après': ap})
    comp['delta_%'] = 100 * (comp['après'] - comp['avant']) / comp['avant'].abs().replace(0, np.nan)
    return comp.round(4)

# Test de fumée sur un vrai fichier pathologique
_demo = train_df[train_df['label_str'] != HEALTHY_LABEL]['filepath'].iloc[0]
print('Marqueurs (exemple pathologique) :')
for k, v in mesurer_marqueurs(_demo).items():
    print(f'  {k:9s}: {v:.4f}')

Marqueurs (exemple pathologique) :
  jitter   : 0.0037
  shimmer  : 0.0050
  hnr      : 33.2847
  f0_mean  : 77.0600
  f0_std   : 0.8693


## 4. Augmentation signal-level (hors-ligne) + rééquilibrage

Faute de wheels disponibles pour `audiomentations`/`librosa` sous Python 3.14,
les transformations signal-level sont implémentées directement en **NumPy /
torchaudio / SciPy** — ce qui les rend totalement transparentes et contrôlables.
Toutes sont **additives ou convolutives** (bruit, gain, réverbération de salle
légère) : elles simulent des conditions de microphone/pièce **sans toucher à la
source glottique**, donc préservent en principe le label (vérifié en fin de cellule).

On génère des variantes pour amener chaque classe pathologique à `TARGET_PER_CLASS`
exemples d'entraînement. La classe `Healthy` est **complétée par de vrais
locuteurs SVD** (le pool en compte ~687) plutôt qu'augmentée artificiellement.

> Les augmentations ne sont générées qu'à partir de patients du **train**, et
> héritent du `Last_Name` source → aucune fuite possible vers la validation.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 4 — Augmentation signal-level hors-ligne + rééquilibrage
# ═══════════════════════════════════════════════════════════════════════════
from scipy.signal import fftconvolve

def _rms(x):
    return float(np.sqrt(np.mean(x ** 2) + 1e-12))

def aug_gaussian_snr(wav, snr_db, rng):
    noise = rng.standard_normal(len(wav)).astype('float32')
    target = _rms(wav) / (10 ** (snr_db / 20))
    return wav + noise * (target / (_rms(noise) + 1e-12))

def aug_gain(wav, gain_db):
    return (wav * (10 ** (gain_db / 20))).astype('float32')

def aug_reverb(wav, sr, decay_s, rng):
    n = max(8, int(sr * decay_s))
    ir = (rng.standard_normal(n) * np.exp(-np.linspace(0, 6, n))).astype('float32')
    ir[0] = 1.0
    out = fftconvolve(wav, ir)[:len(wav)].astype('float32')
    peak = np.max(np.abs(out)) + 1e-9
    return out / peak * (np.max(np.abs(wav)) + 1e-9)

def augmenter_signal(wav, sr, rng):
    "Applique 1 a 2 transformations legeres tirèes aléatoirement."
    ops = rng.permutation(['noise', 'gain', 'reverb'])[:rng.integers(1, 3)]
    for op in ops:
        if op == 'noise':
            wav = aug_gaussian_snr(wav, snr_db=float(rng.uniform(20, 40)), rng=rng)
        elif op == 'gain':
            wav = aug_gain(wav, gain_db=float(rng.uniform(-3, 3)))
        elif op == 'reverb':
            wav = aug_reverb(wav, sr, decay_s=float(rng.uniform(0.08, 0.20)), rng=rng)
    m = np.max(np.abs(wav))
    if m > 1.0:
        wav = wav / m
    return wav.astype('float32')

# ── Génération équilibrante pour les classes pathologiques ───────────────────
signal_rows = []
for cls in TARGET_PATHOLOGIES:
    sub = train_df[train_df['label_str'] == cls]
    n_have = len(sub)
    n_need = max(0, TARGET_PER_CLASS - n_have)
    if n_need == 0:
        continue
    srcs = sub.sample(n=n_need, replace=True, random_state=SEED).reset_index(drop=True)
    for i, row in srcs.iterrows():
        wav = _load_mono_16k(row['filepath']).astype('float32')
        aug = augmenter_signal(wav, TARGET_SR, RNG)
        fname = f"{cls.replace(' ', '_')}_{i:04d}_sig.wav"
        fpath = os.path.join(AUG_SIGNAL_DIR, fname)
        sf.write(fpath, aug, TARGET_SR)
        signal_rows.append({'filepath': fpath, 'File_Name': fname,
                            'Last_Name': row['Last_Name'], 'label_str': cls,
                            'label': label2id[cls], 'is_aug': True,
                            'aug_type': 'signal', 'source_filepath': row['filepath']})
signal_aug_df = pd.DataFrame(signal_rows)
print(f'Variantes signal-level générées : {len(signal_aug_df)}')

# ── Complément Healthy par de vrais locuteurs SVD (hors validation) ──────────
val_names = set(val_df['Last_Name'])
used = set(df['filepath'])
all_svd = glob.glob(os.path.join(SVD_HEALTHY_DIR, '*', 'vowels', '*a_h.wav'))
candidats = [f for f in all_svd
             if f not in used
             and os.path.basename(os.path.dirname(os.path.dirname(f))) not in val_names]
n_health_have = (train_df['label_str'] == HEALTHY_LABEL).sum()
n_health_need = max(0, TARGET_PER_CLASS - n_health_have)
random.seed(SEED)
extra = random.sample(candidats, min(n_health_need, len(candidats)))
healthy_rows = [{'filepath': f, 'File_Name': os.path.basename(f),
                 'Last_Name': os.path.basename(os.path.dirname(os.path.dirname(f))),
                 'label_str': HEALTHY_LABEL, 'label': label2id[HEALTHY_LABEL],
                 'is_aug': False, 'aug_type': 'real_extra', 'source_filepath': f}
                for f in extra]
healthy_extra_df = pd.DataFrame(healthy_rows)
print(f'Locuteurs sains réels ajoutés    : {len(healthy_extra_df)}')

# ── Vérification parselmouth : le signal-level préserve-t-il les marqueurs ? ──
if len(signal_aug_df):
    ech = signal_aug_df.sample(min(8, len(signal_aug_df)), random_state=SEED)
    avant = list(ech['source_filepath'])
    apres = [_load_mono_16k(p) for p in ech['filepath']]
    print('\nPréservation des marqueurs (signal-level, moyenne sur échantillon) :')
    print(comparer_marqueurs(avant, apres).to_string())

Variantes signal-level générées : 123
Locuteurs sains réels ajoutés    : 0

Préservation des marqueurs (signal-level, moyenne sur échantillon) :
            avant     après  delta_%
jitter     0.0298    0.0303   1.5262
shimmer    0.0853    0.0789  -7.4008
hnr       14.3433   12.9308  -9.8475
f0_mean  160.6920  177.5758  10.5069
f0_std     7.4097   10.3461  39.6280


## 5. Conversion de voix SeedVC (hors-ligne, expérimental)

**Objectif :** convertir le **timbre** d'un enregistrement pathologique vers de
nouveaux locuteurs (références SVD), afin d'augmenter la **diversité de voix** par
classe et réduire le sur-apprentissage à des patients précis — *tout en conservant
le contenu et la prosodie pathologique* grâce à la variante **conditionnée par F0**.

**Risque majeur :** la conversion peut « normaliser » la voix et **effacer la
pathologie** (le label deviendrait faux). C'est pourquoi chaque conversion passe le
**gate parselmouth** : si jitter/shimmer/HNR s'effondrent vers des valeurs saines,
on **rejette** le tier VC.

> ⚙️ **Installation** — SeedVC (dépôt `Plachtaa/seed-vc`) possède une arborescence
> de dépendances lourde, parfois incompatible avec Python 3.14. La cellule est donc
> **défensive** : si SeedVC n'est pas disponible, le tier VC est *proprement ignoré*
> et l'expérience E3 est sautée (E2 reste la meilleure configuration retenue).
> Génération **hors-ligne** : on peut aussi peupler `AUG_VC_DIR` par tout autre
> moyen (env. séparé, CLI SeedVC) puis relancer la Cellule 6.

In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 5 — SeedVC (conversion de voix conditionnée F0) — défensif
# ═══════════════════════════════════════════════════════════════════════════
SEEDVC_ENABLE = True                                  # passer à False pour ignorer
SEEDVC_DIR    = r'C:/Users/AdminIA/Documents/models/seed-vc'   # dépôt cloné (si présent)
N_VC_PER_SRC  = 1                                     # nb de timbres cibles / source

VC_AVAILABLE = False
convert_seedvc = None
if SEEDVC_ENABLE:
    try:
        import sys
        if os.path.isdir(SEEDVC_DIR):
            sys.path.insert(0, SEEDVC_DIR)
        # ── Point d'intégration SeedVC ───────────────────────────────────────
        # L'API exacte dépend de la version de SeedVC. Brancher ici l'appel réel,
        # p.ex. via le module d'inférence du dépôt, en RENVOYANT une forme d'onde
        # 16 kHz mono (np.float32). Exemple de signature attendue :
        #     def convert_seedvc(source_wav_16k, reference_path) -> np.ndarray
        # Tant qu'il n'est pas branché, on lève une exception → tier VC ignoré.
        raise ImportError("Brancher l'appel d'inférence SeedVC ici.")
        VC_AVAILABLE = True
    except Exception as e:
        print('SeedVC indisponible → tier VC (E3) ignoré.')
        print('  Détail :', e)
        print('  Pour activer : cloner Plachtaa/seed-vc, télécharger les checkpoints')
        print('  (modèle conditionné F0), puis brancher convert_seedvc(...) ci-dessus.')

vc_rows = []
if VC_AVAILABLE:
    # Pool de timbres de référence = locuteurs SVD sains (hors validation)
    val_names = set(val_df['Last_Name'])
    ref_pool = [f for f in glob.glob(os.path.join(SVD_HEALTHY_DIR, '*', 'vowels', '*a_h.wav'))
                if os.path.basename(os.path.dirname(os.path.dirname(f))) not in val_names]
    rng_vc = np.random.default_rng(SEED)
    patho = train_df[train_df['label_str'] != HEALTHY_LABEL]
    converted_wavs, converted_srcs = [], []
    for _, row in patho.iterrows():
        src = _load_mono_16k(row['filepath']).astype('float32')
        for k in range(N_VC_PER_SRC):
            ref = ref_pool[int(rng_vc.integers(0, len(ref_pool)))]
            try:
                out = convert_seedvc(src, ref)
            except Exception as e:
                print('Conversion échouée :', e); continue
            fname = f"{row['label_str'].replace(' ', '_')}_{row['Last_Name']}_{k}_vc.wav"
            fpath = os.path.join(AUG_VC_DIR, fname)
            sf.write(fpath, np.asarray(out, dtype='float32'), TARGET_SR)
            vc_rows.append({'filepath': fpath, 'File_Name': fname,
                            'Last_Name': row['Last_Name'], 'label_str': row['label_str'],
                            'label': label2id[row['label_str']], 'is_aug': True,
                            'aug_type': 'seedvc', 'source_filepath': row['filepath']})
            converted_wavs.append(np.asarray(out, dtype='float32'))
            converted_srcs.append(row['filepath'])

    # ── GATE parselmouth : la pathologie survit-elle à la conversion ? ───────
    if converted_wavs:
        comp = comparer_marqueurs(converted_srcs[:12], converted_wavs[:12])
        print('\nGate VC — marqueurs source vs converti :')
        print(comp.to_string())
        # Heuristique : si jitter & shimmer chutent de >50 %, la pathologie est
        # probablement effacée → on rejette le tier VC.
        d_j = comp.loc['jitter', 'delta_%'] if 'jitter' in comp.index else 0
        d_s = comp.loc['shimmer', 'delta_%'] if 'shimmer' in comp.index else 0
        if (d_j is not None and d_j < -50) or (d_s is not None and d_s < -50):
            print('\n⚠ Effondrement des marqueurs → tier VC REJETÉ (on garde E2).')
            vc_rows = []
        else:
            print('\n✓ Marqueurs préservés → tier VC RETENU pour E3.')

vc_aug_df = pd.DataFrame(vc_rows)
print(f'\nConversions SeedVC retenues : {len(vc_aug_df)}')

SeedVC indisponible → tier VC (E3) ignoré.
  Détail : Brancher l'appel d'inférence SeedVC ici.
  Pour activer : cloner Plachtaa/seed-vc, télécharger les checkpoints
  (modèle conditionné F0), puis brancher convert_seedvc(...) ci-dessus.

Conversions SeedVC retenues : 0


## 6. Assemblage des manifestes d'entraînement + garde-fous anti-fuite

On construit deux manifestes d'entraînement, en gardant le **`val_df` original
inchangé** :

- **`train_bal`** (pour E1 et E2) = train original + variantes signal-level + sains réels ;
- **`train_bal_vc`** (pour E3) = `train_bal` + conversions SeedVC retenues.

Un `assert` vérifie qu'**aucun patient source** des échantillons augmentés
n'appartient à la validation.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 6 — Assemblage des manifestes + garde-fous
# ═══════════════════════════════════════════════════════════════════════════
cols = ['filepath', 'File_Name', 'Last_Name', 'label_str', 'label',
        'is_aug', 'aug_type', 'source_filepath']
base = train_df.assign(label=train_df['label_str'].map(label2id))[cols]

parts = [base]
if len(signal_aug_df):   parts.append(signal_aug_df[cols])
if len(healthy_extra_df):parts.append(healthy_extra_df[cols])
train_bal = pd.concat(parts, ignore_index=True)

train_bal_vc = train_bal.copy()
if len(vc_aug_df):
    train_bal_vc = pd.concat([train_bal, vc_aug_df[cols]], ignore_index=True)

# ── Garde-fou anti-fuite : aucun patient source augmenté en validation ───────
val_names = set(val_df['Last_Name'])
for name, m in [('train_bal', train_bal), ('train_bal_vc', train_bal_vc)]:
    aug_src_names = set(m.loc[m['is_aug'], 'Last_Name'])
    fuite = aug_src_names & val_names
    assert not fuite, f'[{name}] Fuite via augmentation : {sorted(fuite)}'

# Persistance des manifestes (traçabilité)
train_bal.to_csv(os.path.join(AUG_AUDIO_DIR, 'manifest_train_bal.csv'), index=False)
train_bal_vc.to_csv(os.path.join(AUG_AUDIO_DIR, 'manifest_train_bal_vc.csv'), index=False)

print('Répartition train_bal (E1/E2) :')
print(train_bal['label_str'].value_counts().reindex(CLASSES).to_string())
print(f'  total : {len(train_bal)}')
print('\nRépartition train_bal_vc (E3) :')
print(train_bal_vc['label_str'].value_counts().reindex(CLASSES).to_string())
print(f'  total : {len(train_bal_vc)}')
print('\n✓ Aucune fuite patient détectée. Val (fixe) :', len(val_df))

## 7. Entraînement des expériences (E1 → E3)

Chaque expérience réutilise la boucle de `foch_utils` (AMP, accumulation, cosine,
early stopping). La seule différence contrôlée :

- **E1** `aug_signal` — données rééquilibrées, **SpecAugment désactivé** ;
- **E2** `aug_signal_specaug` — mêmes données, **SpecAugment activé** (masquage
  interne temps/fréquence de WavLM) ;
- **E3** `aug_full` — données + SeedVC (si le tier VC a été retenu), SpecAugment activé.

Le jeu étant désormais équilibré, la perte est **non pondérée**.

> ⏱️ **Coût** — 3 entraînements de WavLM-Large. Pour un test rapide du pipeline,
> réduire `EPOCHS` (ex. 3) avant de lancer l'ablation complète.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 7 — Entraînement des expériences
# ═══════════════════════════════════════════════════════════════════════════
import gc

def run_experiment(run_name, train_manifest, spec_augment):
    print('\n' + '=' * 70)
    print(f'EXPÉRIENCE : {run_name}  | SpecAugment={spec_augment} | '
          f'train={len(train_manifest)}')
    print('=' * 70)
    train_loader, val_loader = F.build_dataloaders(
        train_manifest, val_df, processor, BATCH_SIZE, TARGET_SR, MAX_LEN_S)
    model = F.build_model(MODEL_DIR, NUM_LABELS, label2id, id2label, device,
                          freeze_fe=FREEZE_FE, spec_augment=spec_augment,
                          mask_time_prob=SPECAUG_TIME_PROB,
                          mask_feature_prob=SPECAUG_FEATURE_PROB)
    ckpt = os.path.join(WEIGHTS_DIR, f'{run_name}_best.pt')
    hist = F.train_model(model, train_loader, val_loader, device, ckpt,
                         epochs=EPOCHS, lr=LR, accum_steps=ACCUM_STEPS,
                         patience=PATIENCE, class_weights=None)
    metrics = F.evaluate_model(model, val_loader, CLASSES, device, hist,
                               METRICS_DIR, run_name, best_ckpt_path=ckpt)
    del model, train_loader, val_loader
    gc.collect(); torch.cuda.empty_cache()
    return metrics

# E1 — signal-level seul
m_e1 = run_experiment('aug_signal', train_bal, spec_augment=False)
# E2 — signal-level + SpecAugment
m_e2 = run_experiment('aug_signal_specaug', train_bal, spec_augment=True)
# E3 — + SeedVC (uniquement si le tier VC a été retenu)
if len(vc_aug_df):
    m_e3 = run_experiment('aug_full', train_bal_vc, spec_augment=True)
else:
    print('\nTier VC non disponible → E3 sautée.')

## 8. Ablation comparative

On charge les métriques sauvegardées (`*_metrics.json`) du baseline `2_0` et des
expériences E1–E3, puis on les compare sur les mesures **robustes au déséquilibre**
(balanced accuracy, macro-F1, macro AUC). La décision finale privilégie
l'augmentation qui améliore le **macro-F1** *sans dégrader le rappel des
pathologies* (à vérifier sur les matrices de confusion par classe).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cellule 8 — Ablation comparative
# ═══════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

runs = [('E0 baseline', 'wavlm_large_baseline'),
        ('E1 signal',   'aug_signal'),
        ('E2 +specaug',  'aug_signal_specaug'),
        ('E3 +seedvc',   'aug_full')]

rows = []
for label, rn in runs:
    p = os.path.join(METRICS_DIR, f'{rn}_metrics.json')
    if not os.path.isfile(p):
        continue
    d = json.load(open(p, encoding='utf-8'))
    rows.append({'Expérience': label,
                 'Balanced acc': d.get('balanced_accuracy'),
                 'Macro-F1': d.get('macro_f1'),
                 'Macro AUC': d.get('macro_auc_ovr')})
comp = pd.DataFrame(rows)
print(comp.to_string(index=False))

if len(comp):
    ax = comp.set_index('Expérience')[['Balanced acc', 'Macro-F1', 'Macro AUC']].plot(
        kind='bar', figsize=(11, 6), ylim=(0, 1), rot=0)
    ax.set_title('FOCH — Ablation augmentation (validation originale fixe)')
    ax.set_ylabel('Score'); ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(METRICS_DIR, 'ablation_augmentation.png'), dpi=150, bbox_inches='tight')
    plt.show()
    comp.to_csv(os.path.join(METRICS_DIR, 'ablation_augmentation.csv'), index=False)
    print('\n✓ Comparaison sauvegardée dans', METRICS_DIR)

## 9. Synthèse

Ce notebook ajoute une augmentation de données **rigoureuse et mesurée** au baseline
multi-classes :

1. **Signal-level** (NumPy/torchaudio) — rééquilibrage des pathologies, additif/convolutif,
   à faible risque pour la source glottique (vérifié parselmouth).
2. **SpecAugment** — masquage interne temps/fréquence de WavLM, gratuit et robuste.
3. **SeedVC** (conditionné F0) — diversité de locuteurs, **sous réserve** du gate
   parselmouth qui rejette la conversion si elle efface la pathologie.

**Garanties méthodologiques :** validation 100 % originale et fixe, augmentation
train-only post-split, traçabilité complète (manifestes), garde-fous anti-fuite.

**Lecture des résultats :** comparer E1→E3 à E0 sur le **macro-F1** et le **rappel par
classe** (matrices), en gardant à l'esprit que `fuite glottique` n'a que ~5 exemples
de validation — ses scores sont à interpréter avec prudence (intervalle large).

**Pistes ultérieures :** validation croisée 5 plis (intervalles de confiance),
dégel progressif du transformer, et — si SeedVC s'avère destructeur — explorer une
conversion *préservant davantage la source* (ex. conditionnement prosodique renforcé).